In [3]:
from openai import OpenAI
from qdrant_client import QdrantClient
import os
from dotenv import load_dotenv

load_dotenv(
    "/Users/benny/AI-Engineer/.env",
    override=True,
)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

response = client.embeddings.create(
    input="random text",
    model="text-embedding-3-small",
)

#### Embedding function

In [4]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

#### Retrieval function

In [5]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [8]:
def retrieve_similar_books(query, qdrant_client, top_k=5):

    query_embedding = get_embedding(query)
    
    search_result = qdrant_client.query_points(
        collection_name="amazon_books_collection_sampled",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in search_result.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context.append(result.payload['description'])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload['average_rating'])
    
    return {
        "context_ids": retrieved_context_ids,
        "context": retrieved_context,
        "similarity_scores": similarity_scores,
        "ratings": retrieved_context_ratings
    }
    

In [9]:
retrieved_context = retrieve_similar_books('erotica that can be studied in the academia', qdrant_client)

In [10]:
retrieved_context

{'context_ids': ['B008SLE2F2',
  'B00HR8OIBU',
  'B00AHGKMIC',
  '1593081146',
  '0691067228'],
 'context': ['Romancing Miss Bronte: A NovelIn this astonishing novel, a brilliant mélange of fact and fiction, Juliet Gael skillfully and stylishly captures the passions, hopes, dreams, and sorrows of literature’s most famous sisters—and imagines how love dramatically and most unexpectedly found Charlotte Brontë.During the two years that she studied in Brussels, Charlotte had a taste of life’s splendors—travel, literature, and art. Now, back home in the Yorkshire moors, duty-bound to a blind father and an alcoholic brother, an ambitious Charlotte refuses to sink into hopelessness. With her sisters, Emily and Anne, Charlotte conceives a plan to earn money and pursue a dream: The Brontës will publish. In childhood the Brontë children created fantastical imaginary worlds; now the sisters craft novels quite unlike anything written before. Transforming her loneliness and personal sorrow into a t

#### format retrieved context function


In [16]:
def process_context(retrieved_context):
    formatted_context = ""

    for id, chunk, rating in zip(retrieved_context["context_ids"], retrieved_context["context"], retrieved_context["ratings"]):
        formatted_context += f"- ID: {id}, Rating: {rating}, Description: {chunk}\n\n"
    return formatted_context

In [17]:
processed_context = process_context(retrieved_context)

In [18]:
print(processed_context)

- ID: B008SLE2F2, Rating: 4.5, Description: Romancing Miss Bronte: A NovelIn this astonishing novel, a brilliant mélange of fact and fiction, Juliet Gael skillfully and stylishly captures the passions, hopes, dreams, and sorrows of literature’s most famous sisters—and imagines how love dramatically and most unexpectedly found Charlotte Brontë.During the two years that she studied in Brussels, Charlotte had a taste of life’s splendors—travel, literature, and art. Now, back home in the Yorkshire moors, duty-bound to a blind father and an alcoholic brother, an ambitious Charlotte refuses to sink into hopelessness. With her sisters, Emily and Anne, Charlotte conceives a plan to earn money and pursue a dream: The Brontës will publish. In childhood the Brontë children created fantastical imaginary worlds; now the sisters craft novels quite unlike anything written before. Transforming her loneliness and personal sorrow into a triumph of literary art, Charlotte pens her 1847 masterpiece, Jane 

#### Create prompt function

In [27]:
def build_prompt(preprocessed_context, user_query):
    prompt = f"""
    You are a book recommendation assistant that can answer questions about the books in stock.

    You will be given a question and a list of context.

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as the available products
    
    Context:
    {preprocessed_context}
    
    Question: {user_query}
    
    Answer:
    """
    return prompt

In [20]:
prompt = build_prompt(processed_context, "What are some good books about erotica that can be studied in the academia?")

In [21]:
prompt

"\n    You are a book recommendation assistant that can answer questions about the books in stock.\n\n    Context:\n    - ID: B008SLE2F2, Rating: 4.5, Description: Romancing Miss Bronte: A NovelIn this astonishing novel, a brilliant mélange of fact and fiction, Juliet Gael skillfully and stylishly captures the passions, hopes, dreams, and sorrows of literature’s most famous sisters—and imagines how love dramatically and most unexpectedly found Charlotte Brontë.During the two years that she studied in Brussels, Charlotte had a taste of life’s splendors—travel, literature, and art. Now, back home in the Yorkshire moors, duty-bound to a blind father and an alcoholic brother, an ambitious Charlotte refuses to sink into hopelessness. With her sisters, Emily and Anne, Charlotte conceives a plan to earn money and pursue a dream: The Brontës will publish. In childhood the Brontë children created fantastical imaginary worlds; now the sisters craft novels quite unlike anything written before. Tr

#### Generate answer function  

In [22]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model = 'gpt-5-nano',
        messages =[{"role": "system", "content": prompt}],
        reasoning_effort = 'minimal'
    )

    return response.choices[0].message.content

In [23]:
print(generate_answer(prompt))

Here are some academically respected books about erotica and sexuality that are often used in scholarly study:

- The Hard Facts of the Grimms' Fairy Tales: Expanded Second Edition by Maria Tatar (discusses darker, erotic, and taboo elements in Grimm tales from psychological and historical perspectives)
- La Ronde (Modern Plays) by Arthur Schnitzler (explores serial sexual encounters across social classes; widely studied for its themes of desire, power, and morality)
- Dracula by Bram Stoker (not erotica per se, but many academic treatments analyze sexuality, vampirism, and queerness in its subtext)
- Romancing Miss Brontë by Juliet Gael (historical fiction—may be studied in terms of portrayals of female desire and literary gatekeeping; useful in a broader course on gender and romance in literature)
- Delphi Complete Works of George Eliot (Illustrated) (examines sexuality, marriage, autonomy, and social expectations in Victorian fiction)
- The complete Grimm tales-related scholarship (

#### Combined RAG pipeline

In [28]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url = "http://localhost:6333")

    retrieved_context = retrieve_similar_books(question, qdrant_client, top_k)
    processed_context = process_context(retrieved_context)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt)
    return answer

In [29]:
print(rag_pipeline("What are some good classic with average rating above 4.5?", top_k=5))

Here are a few classics with average rating above 4.5 from the available products:
- ID 0451527291: Four Great Tragedies: Hamlet, Othello, King Lear, Macbeth (Signet Classics) — Rating 4.6
- ID 034544597X: Bittersweet by Freddie Lee Johnson III — Rating 4.6
- ID 1593081146: Dracula (Barnes & Noble Classics) — Rating 4.5 (Note: exactly 4.5, not above) 
- ID 1607103109: War and Peace — Rating 4.4 (below 4.5)

If you want strictly above 4.5, stick with:
- Four Great Tragedies (Rating 4.6)
- Bittersweet (Rating 4.6)


In [30]:
print(rag_pipeline("I like to read some science fiction with average rating above 4.5"))

Here are science fiction titles from the available products with average rating above 4.5:

- The Fantasies of Robert A. Heinlein (ID: 0312872453) — Rating: 4.5 (not above 4.5, so exclude)
- The Mammoth Book of Paranormal Romance (ID: 0762436514) — Rating: 3.9 (not SF, and below 4.5)
- The Good New Stuff: Adventure in SF in the Grand Tradition (ID: 0312198906) — Rating: 4.2
- The Year's Best Science Fiction: Twenty-Eighth Annual Collection (ID: 0312546335) — Rating: 4.2
- Wireless (ID: 0441017193) — Rating: 4.3

All listed SF titles have ratings 4.3 or below, none above 4.5. If you’re flexible with rating thresholds or genres adjacent to SF, I can suggest other available options.
